# Compare Chunking Outputs

- Loads `chunker.ts`, `chunker-semantic.ts`, and legacy Python JSON outputs.
- Uses pandas tables so differences are easier to inspect.
- Goal: make the new TS semantic logic behave more like the known working Python legacy logic.


In [1]:
#Imports
from __future__ import annotations
import json
from pathlib import Path
import pandas as pd

BASE_DIR = Path('/Users/matthewplambeck/Desktop/Deployable-Knowledge/outputs-test')
FILES = {
    'baseline_ts': BASE_DIR / 'chunker.json',
    'semantic_ts': BASE_DIR / 'chunker-semantic.json',
    'python_legacy': BASE_DIR / 'chunker-python.json',}

def load_chunks(path: Path, source_name: str) -> pd.DataFrame:
    payload = json.loads(path.read_text())
    rows = payload['chunks'] if isinstance(payload, dict) and 'chunks' in payload else payload
    frame = pd.DataFrame(rows).copy()
    frame['source_name'] = source_name
    if isinstance(payload, dict) and 'model' in payload:
        frame['model'] = payload['model']
    else:
        frame['model'] = source_name
    frame['wordCount'] = frame['content'].str.split().str.len()
    frame['sentenceCountApprox'] = (
        frame['content']
        .str.replace('\n', ' ', regex=False)
        .str.split('.')
        .apply(lambda parts: sum(1 for part in parts if part.strip()))
    )
    return frame[['source_name', 'model', 'pageIndex', 'chunkIndex', 'chunkId', 'wordCount', 'sentenceCountApprox', 'content']]

frames = {name: load_chunks(path, name) for name, path in FILES.items()}
summary_counts = pd.DataFrame({
    name: {
        'chunks': len(frame),
        'pages_with_chunks': frame['pageIndex'].nunique(),
        'mean_words': round(frame['wordCount'].mean(), 2),
        'median_words': frame['wordCount'].median(),
        'max_words': frame['wordCount'].max(),
        'mean_sentences': round(frame['sentenceCountApprox'].mean(), 2),
        'median_sentences': frame['sentenceCountApprox'].median(),
        'max_sentences': frame['sentenceCountApprox'].max(),
    }
    for name, frame in frames.items()
}).T
summary_counts


FileNotFoundError: [Errno 2] No such file or directory: '/Users/matthewplambeck/Desktop/Deployable-Knowledge/outputs-test/chunker.json'

In [ ]:
# Sentence count distribution by implementation
distribution = (
    pd.concat(frames.values(), ignore_index=True)
    .groupby(['source_name', 'sentenceCountApprox'])
    .size()
    .rename('chunkCount')
    .reset_index()
    .pivot(index='sentenceCountApprox', columns='source_name', values='chunkCount')
    .fillna(0)
    .astype(int)
)
distribution


source_name,baseline_ts,python_legacy,semantic_ts
sentenceCountApprox,,,
1,1284,115,89
2,139,74,65
3,30,47,53
4,7,27,43
5,0,28,25
6,0,11,13
7,2,5,12
8,0,9,7
9,0,6,3


## Coverage Check

Compare normalized sentence and word coverage between `semantic_ts` and `python_legacy` so chunk-size differences do not hide missing text.


In [ ]:
import re

def normalize_text(text: str) -> str:
    return re.sub(r'\s+', ' ', text).strip()

def split_sentence_like(text: str) -> list[str]:
    normalized = normalize_text(text)
    if not normalized:
        return []
    parts = re.split(r'(?<=[.!?])\s+', normalized)
    return [part.strip() for part in parts if part.strip()]

def tokenize_words(text: str) -> list[str]:
    return re.findall(r"[A-Za-z0-9']+", text.lower())

def collect_sentence_set(frame: pd.DataFrame) -> set[str]:
    sentences = set()
    for content in frame['content']:
        sentences.update(split_sentence_like(content))
    return sentences

def collect_word_set(frame: pd.DataFrame) -> set[str]:
    words = set()
    for content in frame['content']:
        words.update(tokenize_words(content))
    return words

semantic_sentences = collect_sentence_set(frames['semantic_ts'])
python_sentences = collect_sentence_set(frames['python_legacy'])
semantic_words = collect_word_set(frames['semantic_ts'])
python_words = collect_word_set(frames['python_legacy'])

coverage_summary = pd.DataFrame([
    {
        'metric': 'sentences',
        'semantic_count': len(semantic_sentences),
        'python_count': len(python_sentences),
        'missing_from_semantic': len(python_sentences - semantic_sentences),
        'extra_in_semantic': len(semantic_sentences - python_sentences),
        'semantic_overlap_pct': round(100 * len(semantic_sentences & python_sentences) / max(len(python_sentences), 1), 2),
    },
    {
        'metric': 'words',
        'semantic_count': len(semantic_words),
        'python_count': len(python_words),
        'missing_from_semantic': len(python_words - semantic_words),
        'extra_in_semantic': len(semantic_words - python_words),
        'semantic_overlap_pct': round(100 * len(semantic_words & python_words) / max(len(python_words), 1), 2),
    },
])
coverage_summary

missing_sentence_samples = pd.DataFrame({
    'missing_from_semantic': sorted(python_sentences - semantic_sentences)[:20],
})
extra_sentence_samples = pd.DataFrame({
    'extra_in_semantic': sorted(semantic_sentences - python_sentences)[:20],
})
missing_word_samples = pd.DataFrame({
    'missing_words_from_semantic': sorted(python_words - semantic_words)[:50],
})
extra_word_samples = pd.DataFrame({
    'extra_words_in_semantic': sorted(semantic_words - python_words)[:50],
})

display(coverage_summary)
display(missing_sentence_samples)
display(extra_sentence_samples)
display(missing_word_samples)
display(extra_word_samples)


,metric,semantic_count,python_count,missing_from_semantic,extra_in_semantic,semantic_overlap_pct
0,sentences,796,1005,507,298,49.55
1,words,2728,2922,369,175,87.37


,missing_from_semantic
0,(An acceptable alternative site is located at ...
1,(Available in both the Combat Medic and Tactic...
2,(NOTE: This prevents back flow of blood from t...
3,* * A casualty with TBI (maintain oxygen satur...
4,* * Administer 400 mg moxifloxacin from the CW...
5,* * Administer 400 mg moxifloxacin from the CWPP.
6,* * Allow a conscious casualty to assume any p...
7,* * An unconscious casualty.
8,"* * As the needle enters the pleural space, a ..."
9,* * Assume a C-spine injury until cleared.


,extra_in_semantic
0,( NOTE: This prevents back flow of blood from ...
1,* •• A casualty with TBI (maintain oxygen satu...
2,* •• Administer 400 mg moxifloxacin from the C...
3,* •• Allow a conscious casualty to assume any ...
4,* •• An unconscious casualty.
5,"* •• As the needle enters the pleural space, a..."
6,* •• Assume a C-spine injury until cleared.
7,"* •• Casualty at altitude (over 5,000 feet abo..."
8,* •• Control pain even in an unconscious patie...
9,* •• Cover the eye with a rigid eye shield and...


,missing_words_from_semantic
0,13
1,177
2,1992
3,31
4,34
5,46
6,70
7,928
8,929
9,96


,extra_words_in_semantic
0,101
1,105
2,107
3,109
4,111
5,113
6,115
7,121
8,123
9,55
